# CAAR-CDSS: Kaggle T4 Benchmark Evaluation
**Confidence-Aware Adaptive Retrieval Clinical Decision Support System**

This notebook runs the full MedQA evaluation benchmark on Kaggle using a free NVIDIA T4 GPU (16 GB VRAM).
- **Generation**: `meta-llama/Llama-3.1-8B-Instruct` (fp16 unquantized)
- **RAGAS Judge**: `KagglePipelineJudge` (Reuses loaded fp16 pipeline — **0 API calls**)
- **Verification**: `DeBERTa-v3-large` via Hugging Face Inference API (0 local VRAM)

In [ ]:
# 1. Setup Secrets and Hugging Face Token
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("✅ HF_TOKEN configured successfully.")
except Exception as e:
    print(f"⚠️ Could not fetch HF_TOKEN from Kaggle Secrets: {e}")
    print("Please add 'HF_TOKEN' in Kaggle -> Add-ons -> Secrets.")

# Database URL (PostgreSQL - for production, use SQLite for local testing)
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:////kaggle/working/caar_cdss.db"

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

In [ ]:
# 2. Install Project Dependencies
!pip install -q --upgrade pip
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets chromadb FlagEmbedding ragas
!pip install -q rank-bm25 pydantic pydantic-settings fastapi uvicorn matplotlib pandas
!pip install -q asyncpg psycopg2-binary sqlalchemy alembic python-jose passlib bcrypt python-multipart

In [ ]:
# 3. Clone / Setup Repository Files
import shutil
from pathlib import Path

# If using Kaggle dataset input
repo_input = Path("/kaggle/input/caar-cdss-repo")
if repo_input.exists():
    !cp -r /kaggle/input/caar-cdss-repo/* /kaggle/working/
    print("✅ Copied repo files from Kaggle input dataset.")
else:
    print("ℹ️ Running directly in working directory. Ensure src/ and configs/ are present.")

In [ ]:
# 4. Check GPU Hardware & Auto-Configuration
!python -c "from src.config import detect_hardware; print(detect_hardware())"

In [ ]:
# 5. Ingest Guidelines Corpus (or Mount Pre-built Chroma DB)
# Option A: Stream & index first 5,000 articles (~10-15 min)
!python -m src.cli ingest --corpus epfl-llm/guidelines --limit 5000 --chroma-dir /kaggle/working/chroma_db

# Option B: Full corpus (uncomment for final paper run - takes ~45 min)
# !python -m src.cli ingest --corpus epfl-llm/guidelines --chroma-dir /kaggle/working/chroma_db

# Option C: Mount pre-built Chroma DB as Kaggle Dataset (fastest)
# If you have a pre-built Chroma DB as a Kaggle Dataset, mount it and skip ingestion
# !cp -r /kaggle/input/caar-cdss-chroma/* /kaggle/working/chroma_db/

In [ ]:
# 6. Run Baseline Evaluation Suite on MedQA (N=200 queries, KagglePipelineJudge, 0 API Calls)
!python -m src.experiments.ragas_eval --benchmark medqa --n 200 --mode kaggle_fp16 --judge-model meta-llama/Llama-3.1-8B-Instruct --output /kaggle/working/results/medqa_200_results.json

In [ ]:
# 7. View and Export Results
import json
results_file = Path("/kaggle/working/results/medqa_200_results.json")
if results_file.exists():
    results = json.loads(results_file.read_text())
    print("=== MedQA Benchmark Results ===")
    for metric, score in results.items():
        print(f"{metric}: {score:.4f}")

In [ ]:
# 8. Run Full Evaluation Suite (All Benchmarks + Threshold Sweep)
!python -m src.experiments.runner --benchmarks medqa pubmedqa seeds --n 200 --mode kaggle_fp16 --output /kaggle/working/results/full_eval

In [ ]:
# 9. Run Threshold Sweep for Abstention Curve (Paper Figure 1)
!python -m src.experiments.runner --threshold-sweep --benchmarks seeds --n 8 --mode kaggle_fp16 --output /kaggle/working/results/threshold_sweep

In [ ]:
# 10. Run Embedding Ablation (Paper Table 2)
!python -m src.experiments.runner --embedding-ablation --benchmarks seeds --n 8 --mode kaggle_fp16 --output /kaggle/working/results/embedding_ablation

In [ ]:
# 11. View All Results
import json
from pathlib import Path

results_dir = Path("/kaggle/working/results")
for f in sorted(results_dir.glob("*.json")):
    print(f"\n=== {f.name} ===")
    with open(f) as fp:
        data = json.load(fp)
        if isinstance(data, dict) and "method" in data:
            print(f"  Method: {data['method']}, Acc: {data.get('accuracy', 0):.2f}, Abstain: {data.get('abstention_rate', 0):.2f}, Halluc: {data.get('avg_hallucination', 0):.2f}, AvgK: {data.get('avg_retrieval_k', 0):.1f}")
        elif isinstance(data, dict):
            for k, v in data.items():
                if isinstance(v, dict) and "accuracy" in v:
                    print(f"  {k}: Acc={v['accuracy']:.2f}, Abstain={v.get('abstention_rate', 0):.2f}")

In [ ]:
# 12. Download Results
from google.colab import files
import zipfile
import os

zip_path = "/kaggle/working/results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk("/kaggle/working/results"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/kaggle/working/results")
            zipf.write(file_path, arcname)

files.download(zip_path)
print("Results downloaded!")